In [14]:
import sys
sys.path.append("../src")

from feature_utils import (
    create_sentinel_flags,
    replace_sentinels,
    encode_binary_flags,
    convert_days_columns,
    create_domain_features,
    drop_low_variance,
    aggregate_bureau,
    aggregate_bureau_balance,
    aggregate_previous_application,
    aggregate_pos_cash,
    aggregate_installments,
    aggregate_credit_card,
    build_feature_matrix,

)

In [15]:
import duckdb
import pandas as pd
import seaborn as sns
from pathlib import Path
import numpy as np


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid")

RAW       = Path("../data/raw")
PROCESSED = Path("../data/processed")

con = duckdb.connect()

TABLES = [
    "application_train", "application_test",
    "bureau", "bureau_balance",
    "previous_application", "POS_CASH_balance",
    "installments_payments", "credit_card_balance",
]
for t in TABLES:
    con.execute(f"""
        CREATE OR REPLACE VIEW {t} AS
        SELECT * FROM '{(PROCESSED / f"{t}.parquet").as_posix()}'
    """)

col_desc = pd.read_csv(RAW / "HomeCredit_columns_description.csv",
                       encoding="latin-1", index_col=0)

print("Views:", con.execute("SHOW TABLES").df()["name"].tolist())



Views: ['POS_CASH_balance', 'application_test', 'application_train', 'bureau', 'bureau_balance', 'credit_card_balance', 'installments_payments', 'previous_application']


In [16]:
# ── Load tables ──────────────────────────────────────────────
# Load train and test
df_train = con.execute("SELECT * FROM application_train").df()
df_test = con.execute("SELECT * FROM application_test").df()

# Load child data
df_bureau = con.execute("SELECT * FROM bureau").df()
df_bureau_balance = con.execute("SELECT * FROM bureau_balance").df()
df_prev = con.execute("SELECT * FROM previous_application").df()
df_pos = con.execute("SELECT * FROM POS_CASH_balance").df()
df_inst = con.execute("SELECT * FROM installments_payments").df()
df_cc = con.execute("SELECT * FROM credit_card_balance").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [17]:
# ── Filter to train applicants only ──────────────────────────
train_ids = set(df_train["SK_ID_CURR"])

df_bureau = df_bureau[df_bureau["SK_ID_CURR"].isin(train_ids)]
df_prev   = df_prev[df_prev["SK_ID_CURR"].isin(train_ids)]

# bureau_balance and pos/inst/cc filter via SK_ID_PREV
prev_ids   = set(df_prev["SK_ID_PREV"])
bureau_ids = set(df_bureau["SK_ID_BUREAU"])

df_bureau_balance = df_bureau_balance[df_bureau_balance["SK_ID_BUREAU"].isin(bureau_ids)]
df_pos            = df_pos[df_pos["SK_ID_PREV"].isin(prev_ids)]
df_inst           = df_inst[df_inst["SK_ID_PREV"].isin(prev_ids)]
df_cc             = df_cc[df_cc["SK_ID_PREV"].isin(prev_ids)]

In [18]:
# ── Verify shapes ─────────────────────────────────────────────
for name, df in [("application_train", df_train), ("application_test", df_test),
                  ("bureau", df_bureau), ("bureau_balance", df_bureau_balance),
                  ("previous_application", df_prev), ("POS_CASH_balance", df_pos),
                  ("installments_payments", df_inst), ("credit_card_balance", df_cc)]:
    print(f"{name:<30} {df.shape}")


application_train              (307511, 122)
application_test               (48744, 121)
bureau                         (1465325, 17)
bureau_balance                 (14701612, 3)
previous_application           (1413701, 37)
POS_CASH_balance               (8251754, 8)
installments_payments          (10572221, 8)
credit_card_balance            (2354993, 23)


In [19]:
# Application train transformations
df_train_fe = create_sentinel_flags(df_train)
df_train_fe = replace_sentinels(df_train_fe)
df_train_fe = encode_binary_flags(df_train_fe)
df_train_fe = convert_days_columns(df_train_fe)
df_train_fe = create_domain_features(df_train_fe)
df_train_fe = drop_low_variance(df_train_fe)
print(f"df_train_fe shape: {df_train_fe.shape}")

df_train_fe shape: (307511, 138)


In [23]:
# Child table aggregations
bureau_agg = aggregate_bureau(df_bureau)
bb_agg     = aggregate_bureau_balance(df_bureau_balance, df_bureau)
prev_agg   = aggregate_previous_application(df_prev)
pos_agg    = aggregate_pos_cash(df_pos)
inst_agg   = aggregate_installments(df_inst)
cc_agg     = aggregate_credit_card(df_cc)

# Check shapes
for name, df in [("bureau_agg", bureau_agg), ("bb_agg", bb_agg),
                  ("prev_agg", prev_agg), ("pos_agg", pos_agg),
                  ("inst_agg", inst_agg), ("cc_agg", cc_agg)]:
    print(f"{name:<15} {df.shape}")



bureau_agg      (263491, 23)
bb_agg          (92231, 12)
prev_agg        (291057, 26)
pos_agg         (286967, 12)
inst_agg        (289406, 18)
cc_agg          (77934, 24)


In [24]:
# Assemble feature matrix
df_features = build_feature_matrix(
    df_train_fe, bureau_agg, bb_agg,
    prev_agg, pos_agg, inst_agg, cc_agg
)

print(f"\nFinal feature matrix: {df_features.shape}")
print(f"Rows: {df_features.shape[0]:,}")
print(f"Columns: {df_features.shape[1]:,}")
print(f"\nNew columns added from child tables: {df_features.shape[1] - df_train_fe.shape[1]}")

df_features.to_parquet("../data/processed/feature_matrix_train.parquet", index=False)
print("Feature matrix saved successfully")

Feature matrix shape: (307511, 249)
Total features: 247

Final feature matrix: (307511, 249)
Rows: 307,511
Columns: 249

New columns added from child tables: 111
Feature matrix saved successfully


In [25]:
# Apply same transformations to test
df_test_fe = create_sentinel_flags(df_test)
df_test_fe = replace_sentinels(df_test_fe)
df_test_fe = encode_binary_flags(df_test_fe)
df_test_fe = convert_days_columns(df_test_fe)
df_test_fe = create_domain_features(df_test_fe)
df_test_fe = drop_low_variance(df_test_fe)

# Filter child tables to test IDs
test_ids   = set(df_test["SK_ID_CURR"])
df_bureau_test = con.execute("SELECT * FROM bureau").df()
df_bureau_test = df_bureau_test[df_bureau_test["SK_ID_CURR"].isin(test_ids)]

prev_test  = con.execute("SELECT * FROM previous_application").df()
prev_test  = prev_test[prev_test["SK_ID_CURR"].isin(test_ids)]

prev_ids_test   = set(prev_test["SK_ID_PREV"])
bureau_ids_test = set(df_bureau_test["SK_ID_BUREAU"])

bb_test   = con.execute("SELECT * FROM bureau_balance").df()
bb_test   = bb_test[bb_test["SK_ID_BUREAU"].isin(bureau_ids_test)]

pos_test  = con.execute("SELECT * FROM POS_CASH_balance").df()
pos_test  = pos_test[pos_test["SK_ID_PREV"].isin(prev_ids_test)]

inst_test = con.execute("SELECT * FROM installments_payments").df()
inst_test = inst_test[inst_test["SK_ID_PREV"].isin(prev_ids_test)]

cc_test   = con.execute("SELECT * FROM credit_card_balance").df()
cc_test   = cc_test[cc_test["SK_ID_PREV"].isin(prev_ids_test)]

# Aggregate
bureau_agg_test = aggregate_bureau(df_bureau_test)
bb_agg_test     = aggregate_bureau_balance(bb_test, df_bureau_test)
prev_agg_test   = aggregate_previous_application(prev_test)
pos_agg_test    = aggregate_pos_cash(pos_test)
inst_agg_test   = aggregate_installments(inst_test)
cc_agg_test     = aggregate_credit_card(cc_test)

# Build test feature matrix
df_features_test = build_feature_matrix(
    df_test_fe, bureau_agg_test, bb_agg_test,
    prev_agg_test, pos_agg_test, inst_agg_test, cc_agg_test
)

# Save
df_features_test.to_parquet("../data/processed/feature_matrix_test.parquet", index=False)
print(f"Test feature matrix saved: {df_features_test.shape}")

Feature matrix shape: (48744, 248)
Total features: 246
Test feature matrix saved: (48744, 248)
